# ScholarAI v3 SOTA: Gemma 4 E4B Backend
This notebook deploys the **Deep Semantic Evasion** engine. It uses **AMR Graph Parsing** to strip AI syntax and **Contrastive Decoding (Gemma 4 E4B vs GPT-2)** to generate human-like academic text.

### ⚠️ Prerequisites
Before running, click the **Key icon (Secrets)** on the left and add:
1. `HF_TOKEN`: Your HuggingFace token (must have access to Gemma 4).
2. `NGROK_TOKEN`: Your ngrok authentication token.
3. **Enable "Notebook access"** for both.

## Step 1: Environment & Authentication

In [ ]:
# 1. Reset and Clone
%cd /content
!rm -rf sensorspine-humaniser-v3
!git clone https://github.com/NandishSinha1403/sensorspine-humaniser-v3.git

# 2. Install Dependencies
!pip install --upgrade amrlib fastapi uvicorn "pydantic<=2.12.3" torch torchvision accelerate bitsandbytes python-multipart pyngrok penman unidecode huggingface_hub sentencepiece protobuf "numpy<2.1" "pillow<12.0"
!pip install git+https://github.com/huggingface/transformers.git

# 3. Automated Authentication
from google.colab import userdata
from huggingface_hub import login
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Successfully authenticated with HuggingFace.")
except Exception as e:
    print(f"Authentication Error: {e}. Check your Colab Secrets for 'HF_TOKEN'.")

## Step 2: Deploy AMR Models
Downloads the verified Transformer-based models for semantic parsing and generation.

In [ ]:
%cd /content/sensorspine-humaniser-v3/backend
!mkdir -p models

print("Downloading AMR Parsing Model (STOG)...")
!wget -q --show-progress https://github.com/bjascob/amrlib-models/releases/download/parse_xfm_bart_base-v0_1_0/model_parse_xfm_bart_base-v0_1_0.tar.gz
!tar -xzf model_parse_xfm_bart_base-v0_1_0.tar.gz -C models/
!rm model_parse_xfm_bart_base-v0_1_0.tar.gz
!mv models/model_parse_xfm_bart_base-v0_1_0 models/model_stog

print("\nDownloading AMR Generation Model (GTOS)...")
!wget -q --show-progress https://github.com/bjascob/amrlib-models/releases/download/model_generate_t5wtense-v0_1_0/model_generate_t5wtense-v0_1_0.tar.gz
!tar -xzf model_generate_t5wtense-v0_1_0.tar.gz -C models/
!rm model_generate_t5wtense-v0_1_0.tar.gz
!mv models/model_generate_t5wtense-v0_1_0 models/model_gtos

print("\nAMR Pipeline Ready.")

## Step 3: Start Tunnel & Backend Server
This cell will stay running. Use the printed URL in your local Next.js frontend.

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

try:
    # Set up ngrok tunnel
    ngrok_token = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(ngrok_token)
    
    # Close existing tunnels
    ngrok.kill()
    
    # Connect to port 8000
    public_url = ngrok.connect(8000).public_url
    print(f"\n🚀 BACKEND ACTIVE")
    print(f"COPY THIS URL TO FRONTEND: {public_url}")
    print("================================================\n")
    
    # Start FastAPI
    !python main.py
except Exception as e:
    print(f"Failed to start tunnel: {e}. Check your Colab Secrets for 'NGROK_TOKEN'.")